[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gnoejh/AIBookGitHub/blob/main/15_communication.ipynb)

# Layer 6: Communication - Agent-to-Agent Messaging (A2A)



## Position in the AI System Hierarchy

| Layer | Name | Description |
|-------|------|-------------|
| **Layer 6** | COMMUNICATION **(THIS)** | Formatting, messaging protocols, A2A |
| **Layer 5** | CAPABILITY INTEGRATION | MCP, tools, external systems |
| **Layer 4** | COORDINATION | Multi-agent orchestration |
| **Layer 3** | MEMORY | State management |
| **Layer 2** | INVOCATION | API abstraction |
| **Layer 1** | EXECUTION | Model providers |

**Layer 6** enables agents to communicate, coordinate, and exchange messages through standardized protocols. A2A (Agent-to-Agent Messaging) provides the foundation for multi-agent collaboration.

---

## Learning Path Overview

This notebook follows a structured learning path:

1. **Theory & Concepts** (6.1-6.2): Understand what A2A is and why it exists
2. **Message Envelope** (6.3): Learn the standard message format
3. **Communication Patterns** (6.4): Direct, brokered, and pub/sub patterns
4. **Hands-on Learning** (6.5-6.7): Build a simple in-memory broker
5. **A2A Ecosystem** (6.8): Explore production message brokers and services
6. **Cloud Patterns** (6.9): Google Cloud Pub/Sub and other cloud services
7. **Security & Best Practices** (6.10): Trust, authentication, and reliability
8. **Summary & Comparison** (6.11): Review key concepts and compare with MCP

---

## 6.1 Theoretical Foundation: A2A Architecture

### 6.1.1 Why A2A?

A2A enables autonomous agents to collaborate through asynchronous message exchange:

- **Separate processes, hosts, or languages**: Agents can run independently
- **Trust boundaries and auditing**: Messages include provenance and signatures
- **Specialization**: Different agent roles (planner, researcher, critic, tools)
- **Scalability and parallelism**: Horizontal scaling through message queues
- **Decoupling**: Agents don't need direct knowledge of each other
- **Fault tolerance**: Message persistence enables recovery from failures

### 6.1.2 Core Theoretical Concepts

| Concept | Definition | Role in Stack |
|---------|------------|---------------|
| **Envelope** | Structured message wrapper with metadata | Message layer |
| **Broker** | Central message router delivering to inboxes | Infrastructure layer |
| **Pub/Sub** | Topic-based publish/subscribe pattern | Infrastructure layer |
| **Agent** | Autonomous reasoning unit sending/receiving messages | Application layer |
| **Topic** | Named channel for pub/sub messaging | Routing layer |
| **Queue** | FIFO buffer for point-to-point delivery | Storage layer |
| **Provenance** | Message origin, signature, and audit trail | Security layer |

### 6.1.3 Communication Flow (Theoretical)

```mermaid
sequenceDiagram
    participant A1 as Agent 1<br/>(Planner)
    participant Broker as Message Broker
    participant A2 as Agent 2<br/>(Researcher)
    
    A1->>Broker: envelope(sender: planner, recipient: researcher)
    Broker->>A2: deliver envelope
    A2->>Broker: envelope(sender: researcher, recipient: planner)
    Broker->>A1: deliver envelope
```

---

## 6.2 Message Envelope: Standard Format

A robust envelope enables routing, provenance, and auditing.

### 6.2.1 Envelope Structure

```json
{
  "id": "uuid",
  "ts": "ISO-8601",
  "sender": "agent-id",
  "recipient": "agent-id" | "topic:*",
  "type": "text" | "json" | "tool" | "control",
  "content": "...",
  "context": { 
    "topic": "...", 
    "round": 2,
    "conversation_id": "uuid"
  },
  "provenance": { 
    "hash": "...", 
    "sig": "...",
    "parent_id": "uuid"
  }
}
```

### 6.2.2 Field Descriptions

| Field | Type | Purpose |
|-------|------|---------|
| `id` | UUID | Unique message identifier for deduplication |
| `ts` | ISO-8601 | Timestamp for ordering and auditing |
| `sender` | string | Source agent identifier |
| `recipient` | string \| topic | Target agent or topic pattern |
| `type` | enum | Message type (text, json, tool, control) |
| `content` | any | Message payload (string, JSON, binary) |
| `context` | object | Conversation context and metadata |
| `provenance` | object | Security and audit information |

---

## 6.3 Communication Patterns

A2A supports multiple communication patterns for different use cases:

### 6.3.1 Direct Communication

**Point-to-point** messaging between two agents via HTTP/WebSocket.

```mermaid
graph LR
    A1[Agent 1] -->|HTTP/WebSocket| A2[Agent 2]
    A2 -->|response| A1
    
    style A1 fill:#e1f5ff,color:#000000
    style A2 fill:#e1f5ff,color:#000000
```

**Use cases**: Low-latency, tight coupling, simple topologies

### 6.3.2 Brokered Communication

**Central broker** routes messages to agent inboxes (queues).

```mermaid
graph TB
    A1[Agent 1] -->|send| Broker[Message Broker]
    A2[Agent 2] -->|send| Broker
    A3[Agent 3] -->|send| Broker
    
    Broker -->|deliver| A1
    Broker -->|deliver| A2
    Broker -->|deliver| A3
    
    style Broker fill:#fff4e1,color:#000000
    style A1 fill:#e1f5ff,color:#000000
    style A2 fill:#e1f5ff,color:#000000
    style A3 fill:#e1f5ff,color:#000000
```

**Use cases**: Decoupling, reliability, complex routing

### 6.3.3 Pub/Sub Communication

**Topic-based** publish/subscribe with fan-out to multiple subscribers.

```mermaid
graph TB
    Publisher[Publisher Agent] -->|publish to topic| Topic[Topic: research]
    Topic -->|fan-out| Sub1[Subscriber 1]
    Topic -->|fan-out| Sub2[Subscriber 2]
    Topic -->|fan-out| Sub3[Subscriber 3]
    
    style Topic fill:#e8f5e9,color:#000000
    style Publisher fill:#e1f5ff,color:#000000
    style Sub1 fill:#e1f5ff,color:#000000
    style Sub2 fill:#e1f5ff,color:#000000
    style Sub3 fill:#e1f5ff,color:#000000
```

**Use cases**: Broadcasting, event-driven architectures, loose coupling

---

## 6.4 Hands-on Learning: In-Memory Broker

We'll implement a simple async broker with per-agent queues to understand A2A fundamentals.

In [ ]:
import asyncio, uuid, time
from dataclasses import dataclass, asdict
from typing import Dict, Any, Optional

@dataclass
class Envelope:
    id: str
    ts: float
    sender: str
    recipient: str
    type: str
    content: Any
    context: Optional[Dict[str, Any]] = None
    provenance: Optional[Dict[str, Any]] = None

class Broker:
    def __init__(self):
        self.queues: Dict[str, asyncio.Queue] = {}
    
    def register(self, agent_id: str):
        self.queues.setdefault(agent_id, asyncio.Queue())
    
    async def send(self, env: Envelope):
        q = self.queues.get(env.recipient)
        if not q:
            # drop or dead-letter
            return False
        await q.put(env)
        return True
    
    async def recv(self, agent_id: str, timeout: float = 10.0) -> Optional[Envelope]:
        q = self.queues.setdefault(agent_id, asyncio.Queue())
        try:
            env = await asyncio.wait_for(q.get(), timeout=timeout)
            return env
        except asyncio.TimeoutError:
            return None

class A2AAgent:
    def __init__(self, agent_id: str, broker: Broker):
        self.id = agent_id
        self.broker = broker
        broker.register(agent_id)
    
    async def send(self, recipient: str, type_: str, content: Any, context=None):
        env = Envelope(
            id=str(uuid.uuid4()), ts=time.time(), sender=self.id, recipient=recipient,
            type=type_, content=content, context=context or {}
        )
        ok = await self.broker.send(env)
        return ok
    
    async def receive(self, timeout=5.0) -> Optional[Envelope]:
        return await self.broker.recv(self.id, timeout=timeout)

async def demo():
    broker = Broker()
    planner = A2AAgent('planner', broker)
    researcher = A2AAgent('researcher', broker)
    
    # Planner sends a plan
    await planner.send('researcher', 'text', 'Plan: 1) gather facts; 2) compare; 3) summarize',
                     context={'topic': 'AI safety'})
    # Researcher receives and replies
    env = await researcher.receive()
    print('Researcher received:', asdict(env))
    await researcher.send('planner', 'text', 'Facts: [privacy, bias, alignment]')
    
    env2 = await planner.receive()
    print('Planner received:', asdict(env2))

await demo()

### 6.4.1 Delivery Semantics

Message delivery guarantees:

- **At-most-once** (no retry) — simple and fast, may lose messages
- **At-least-once** (retry with ids) — duplicates possible, dedupe by id
- **Exactly-once** — requires idempotency and coordination

### 6.4.2 Add ACK/NACK

Extend `Broker.send` to return delivery receipt and add `ack` flow to confirm processing.

### 6.4.3 Broker vs Pub/Sub

**Broker** (point-to-point) routes to specific recipients. **Pub/Sub** fans-out to subscribers of a topic.

You can map recipients like `topic:research` and have the broker deliver to all subscribers.

| Pattern | Routing | Use Case |
|---------|---------|----------|
| **Broker** | Point-to-point, specific recipient | Direct agent-to-agent communication |
| **Pub/Sub** | Topic-based, multiple subscribers | Broadcasting, event-driven workflows |

### 6.4.4 WebSocket Implementation (Optional)

When you want cross-host processes, use WebSocket. With `websockets` or `flask-socketio`,
agents connect, authenticate, and exchange envelopes. This is a conceptual sketch:

In [ ]:
# Sketch (not run by default)
# from flask import Flask
# from flask_socketio import SocketIO, emit, join_room
# app = Flask(__name__)
# sio = SocketIO(app, cors_allowed_origins='*')
# @sio.on('register')
# def register(data):
#     agent_id = data['agent_id']
#     join_room(agent_id)
#     emit('registered', {'ok': True})
# @sio.on('send')
# def send(data):
#     recipient = data['recipient']
#     emit('message', data['envelope'], to=recipient)
# if __name__ == '__main__':
#     sio.run(app, host='0.0.0.0', port=6060)

## 6.5 Security & Trust

Production A2A systems require robust security:

- **Identity and auth**: Tokens, mTLS, service accounts
- **Provenance fields and signatures**: Message origin verification
- **Rate limits and quotas**: Prevent abuse and ensure fairness
- **Content filters/guardrails**: Validate message content
- **Audit logging**: Track all message exchanges for compliance

## 6.6 Practice Exercises

1. Add `ack` messages and retry logic for at-least-once delivery.
2. Implement pub/sub topics; allow multiple subscribers to `topic:research`.
3. Add a `critic` agent that listens to all messages and emits feedback envelopes.
4. Add a JSON schema validator for envelope content using `pydantic`.
5. Encrypt content field with `cryptography` for an untrusted broker.
6. Bridge A2A to MCP: if content starts with `TOOL(...)`, invoke the MCP-like server from notebook 5 and return the result.

---

## 6.7 A2A Ecosystem: Brokers, Hubs, and Services

### 6.7.1 Overview

The A2A ecosystem consists of **message brokers**, **pub/sub services**, and **managed platforms** that enable production-grade agent communication.

### 6.7.2 Cloud-Native Pub/Sub Services

| Provider | Service | Type | Key Features |
|----------|---------|------|--------------|
| **Google Cloud** | Cloud Pub/Sub | Managed Pub/Sub | Global scale, at-least-once delivery, filtering |
| **AWS** | SQS / SNS | Managed Queue/Topic | Simple, scalable, serverless |
| **Azure** | Service Bus | Managed Broker | Enterprise features, transactions |
| **Redis** | Pub/Sub | In-memory | Low latency, real-time |
| **NATS** | NATS Streaming | Lightweight | High performance, simple |

### 6.7.3 Enterprise Message Brokers

| Broker | Type | Use Case |
|--------|------|----------|
| **RabbitMQ** | AMQP broker | Complex routing, reliability |
| **Apache Kafka** | Distributed log | High throughput, event streaming |
| **Apache Pulsar** | Unified messaging | Multi-tenancy, geo-replication |
| **Apache RocketMQ** | Distributed queue | Alibaba-scale, transactions |

### 6.7.4 A2A Ecosystem Architecture

```mermaid
graph TB
    subgraph Agents["A2A Agents"]
        A1[Planner Agent]
        A2[Researcher Agent]
        A3[Critic Agent]
        A4[Tool Agent]
    end
    
    subgraph Brokers["Message Brokers & Services"]
        GC[Google Cloud Pub/Sub]
        AWS[AWS SQS/SNS]
        AZ[Azure Service Bus]
        RMQ[RabbitMQ]
        KF[Apache Kafka]
        NATS[NATS]
    end
    
    subgraph Patterns["Communication Patterns"]
        P2P[Point-to-Point]
        PUB[Pub/Sub]
        STREAM[Streaming]
    end
    
    Agents -->|envelopes| Brokers
    Brokers --> Patterns
    Patterns --> Agents
    
    style Agents fill:#e1f5ff,color:#000000
    style Brokers fill:#fff4e1,color:#000000
    style Patterns fill:#e8f5e9,color:#000000
```

### 6.7.5 Choosing the Right Broker

| Requirement | Recommended Solution |
|-------------|---------------------|
| **Low latency, real-time** | Redis Pub/Sub, NATS |
| **High throughput** | Apache Kafka, Apache Pulsar |
| **Cloud-native, managed** | Google Cloud Pub/Sub, AWS SQS/SNS |
| **Complex routing** | RabbitMQ, Azure Service Bus |
| **Simple, lightweight** | NATS, Redis |
| **Event streaming** | Apache Kafka, Apache Pulsar |

---

## 6.8 Google Cloud A2A Pattern (Pub/Sub + Vertex AI)

Google Cloud doesn't ship a single "A2A" product, but its managed services (Pub/Sub, Vertex AI, Cloud Run / Functions, Firestore / AlloyDB) compose naturally into a scalable Agent‑to‑Agent communication fabric.

### 6.8.1 Architecture (Managed Cloud Variant)

```mermaid
graph TB
    Planner[Planner Agent] -->|publish| PubSub[Google Cloud Pub/Sub<br/>topic: agent-messages]
    PubSub -->|fan-out| Researcher[Researcher Agent]
    PubSub -->|fan-out| Synthesizer[Synthesizer Agent]
    PubSub -->|fan-out| Critic[Critic Agent]
    
    Researcher --> Vertex[Vertex AI<br/>Model Invocation]
    Synthesizer --> Vertex
    
    style PubSub fill:#fff4e1,color:#000000
    style Planner fill:#e1f5ff,color:#000000
    style Researcher fill:#e1f5ff,color:#000000
    style Synthesizer fill:#e1f5ff,color:#000000
    style Critic fill:#e1f5ff,color:#000000
    style Vertex fill:#e8f5e9,color:#000000
```

### 6.8.2 Why Pub/Sub for A2A?
- Fully managed, horizontal scalability
- At‑least‑once delivery, retention & replay
- Subscription filters let each agent receive only relevant envelopes
- Dead-letter queues for poison messages

### 6.8.3 Mapping Envelope → Pub/Sub
| Envelope Field | Pub/Sub Usage | Notes |
|----------------|---------------|-------|
| id             | Message ID (or attribute) | For idempotency / dedupe |
| sender         | Attribute `sender` | Filtering / auditing |
| recipient      | Attribute `recipient` or content | Use wildcard topics or per-agent subscription filter |
| type           | Attribute `type` | text / tool / control |
| context.round  | Attribute `round` | Numeric routing context |
| content        | Message data (bytes) | UTF-8 JSON or plain text |
| provenance.sig | Attribute `sig` | Optional signature for trust |

### 6.8.4 Pub/Sub Subscription Filter Example
```
recipient = "researcher" OR recipient = "broadcast" OR type = "control"
```

### 6.8.5 Minimal Python Publisher (Vertex AI + Pub/Sub)
This is illustrative; run only if you configure Google Cloud credentials & libs.
```python
# Optional dependencies (not yet in pyproject):
# pip install google-cloud-pubsub google-cloud-aiplatform
import json, uuid, time, asyncio
from google.cloud import pubsub_v1
from vertexai import init, generative_models

PROJECT_ID = "YOUR_GCP_PROJECT"
TOPIC_ID = "agent-messages"
LOCATION = "us-central1"
MODEL_NAME = "gemini-1.5-flash"  # Example Vertex AI Generative Model

init(project=PROJECT_ID, location=LOCATION)
publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path(PROJECT_ID, TOPIC_ID)

def publish_envelope(sender, recipient, type_, content, context=None):
    envelope = {
        "id": str(uuid.uuid4()),
        "ts": time.time(),
        "sender": sender,
        "recipient": recipient,
        "type": type_,
        "content": content,
        "context": context or {}
    }
    future = publisher.publish(
        topic_path,
        data=json.dumps(envelope).encode("utf-8"),
        sender=sender,
        recipient=recipient,
        type=type_
    )
    return future.result()

# Example: planner produces a plan via Vertex AI then publishes
from vertexai.generative_models import GenerativeModel
model = GenerativeModel(MODEL_NAME)
plan_resp = model.generate_content("Plan a 3-step analysis of AI safety concerns")
plan_text = plan_resp.candidates[0].content.parts[0].text
publish_envelope("planner", "researcher", "text", plan_text, {"topic": "ai_safety", "round": 1})
```

### 6.8.6 Minimal Asynchronous Subscriber (Researcher)
```python
from google.cloud import pubsub_v1
import json

subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(PROJECT_ID, "researcher-sub")

def callback(message: pubsub_v1.subscriber.message.Message):
    env = json.loads(message.data.decode("utf-8"))
    print("Received envelope:", env)
    # TODO: Perform model call, publish follow-up
    message.ack()

streaming_pull = subscriber.subscribe(subscription_path, callback=callback)
print("Listening for messages... (Ctrl+C to stop)")
try:
    streaming_pull.result()
except KeyboardInterrupt:
    streaming_pull.cancel()
```

### 6.8.7 Delivery & Scaling Considerations
| Concern | Strategy |
|---------|----------|
| Ordering | Use ordering keys per conversation/session |
| Idempotency | Store processed ids (Redis / Firestore) |
| Replay | Seek subscription to timestamp for reconstruction |
| Poison Messages | Dead-letter topics with max delivery attempts |
| Backpressure | Flow control settings on subscriber |

### 6.8.8 Integrating with Your Orchestrator
1. Local `MultiModelConversation` publishes each agent output to Pub/Sub (envelope).
2. Remote agents (in Cloud Run) subscribe, respond with new envelopes.
3. Local orchestrator optionally filters only final or critic messages to display.
4. Convert internal `conversation_log` into stream processor (append as messages arrive).

### 6.8.9 Hybrid Local + Cloud Pattern

```mermaid
graph LR
    Local[Local Notebook<br/>Orchestrator] -->|publish| PubSub[Pub/Sub Topic]
    PubSub -->|subscribe| Cloud[Cloud Agents]
    Cloud -->|return| PubSub
    PubSub -->|display| Local
    
    style Local fill:#e1f5ff,color:#000000
    style PubSub fill:#fff4e1,color:#000000
    style Cloud fill:#e8f5e9,color:#000000
```

### 6.8.10 Security Notes
- Use service accounts with least privilege
- IAM: restrict topic publish/subscribe per agent role
- Optionally sign envelope hashes (KMS) and include signature attribute
- Enable Cloud Audit Logs for message access traces

### 6.8.11 Exercise Extensions
1. Add a `control` message type to pause/resume agents (subscription filter changes).
2. Implement a `broadcast` recipient — all agents subscribed with a filter `recipient = "broadcast" OR recipient = "<agent>"`.
3. Add a lightweight dedupe cache keyed by envelope id (in-memory or Redis).
4. Introduce a `critic` Cloud Run service that only subscribes to `type = "text"` and publishes `evaluation` messages.
5. Enforce max rounds per topic: subscriber drops messages where `context.round > MAX_ROUNDS`.
6. Add provenance signature using an HMAC and verify before processing.

### 6.8.12 When to Choose Pub/Sub for A2A
| Scenario | Pub/Sub Fit? | Rationale |
|----------|--------------|-----------|
| Small local experiment | ❌ Overkill | In-memory broker cheaper |
| Multi-agent across regions | ✅ Yes | Managed global scale |
| Need replay / audit | ✅ Yes | Retention + logging |
| Low-latency tight loop | ⚠️ Maybe | Consider direct WebSocket |
| Strict ordering across all agents | ❌ Hard | Partition ordering only |

Use this pattern when you evolve from a single-process teaching orchestrator to distributed, resilient agent swarms.


## 6.9 A2A Summary: Systematic Integration

### 6.9.1 A2A in the Complete System Context

A2A serves as the **communication layer** enabling multi-agent coordination:

```mermaid
graph TB
    L6["Layer 6: Communication<br/><b>A2A PROTOCOL</b><br/>Agent-to-agent messaging<br/>Message brokers & pub/sub<br/>Envelope routing & delivery"]
    L5["Layer 5: Capability Integration<br/>MCP tool invocation<br/>embedded in A2A messages"]
    L4["Layer 4: Coordination<br/>Orchestrates A2A workflows<br/>Multi-agent task decomposition"]
    L3["Layer 3: Memory<br/>Stores A2A conversation history"]
    L2["Layer 2: Invocation<br/>Model calls triggered<br/>by A2A messages"]
    L1["Layer 1: Execution<br/>Models execute<br/>responses sent via A2A"]
    
    L6 --> L5
    L5 --> L4
    L4 --> L3
    L3 --> L2
    L2 --> L1
    
    style L6 fill:#ffeb3b,stroke:#f57f17,stroke-width:3px,color:#000000
    style L5 fill:#e3f2fd,color:#000000
    style L4 fill:#e8f5e9,color:#000000
    style L3 fill:#fff3e0,color:#000000
    style L2 fill:#f3e5f5,color:#000000
    style L1 fill:#e0f2f1,color:#000000
```

### 6.9.2 Key Takeaways

1. **Hierarchical Position**: A2A operates at Layer 6 (Communication), enabling agent coordination
2. **Ecosystem Structure**:
   - **Cloud Services**: Google Cloud Pub/Sub, AWS SQS/SNS, Azure Service Bus
   - **Enterprise Brokers**: RabbitMQ, Apache Kafka, Apache Pulsar
   - **Lightweight**: NATS, Redis Pub/Sub
3. **Communication Patterns**: Direct, Brokered, Pub/Sub
4. **Message Envelope**: Standardized format with id, sender, recipient, type, content, context, provenance
5. **Integration with MCP**: A2A messages can contain MCP tool invocations

---

## 6.10 MCP vs A2A: Understanding the Differences

| Dimension | MCP (Model Context Protocol) | A2A (Agent-to-Agent Messaging) |
|-----------|------------------------------|--------------------------------|
| Purpose | Standard tool/resource discovery & invocation | Coordinated reasoning & workflow across autonomous agents |
| Interaction Mode | Structured JSON-RPC calls | Free-form envelopes / events |
| Transport | stdio / WebSocket | Pub/Sub / WebSocket / HTTP / in-memory |
| Message Shape | Fixed RPC envelope | Flexible agent envelope |
| Discovery | Built-in (`tools/list`, `resources/list`) | External registry or topic filters |
| Negotiation | Yes (`initialize`) | Usually manual / implicit |
| Param Validation | JSON Schema | Optional custom schema (pydantic) |
| Error Semantics | Standard RPC error | App-specific (ACK, retry, DLQ) |
| Streaming Support | Extensions (tool call streaming) | Depends on channel (WebSocket stream, chunked) |
| Scaling Vector | Add more MCP servers/tools | Add more agents & topics/subscriptions |
| Security Focus | Tool scope, resource permission | Identity, provenance, rate limiting, encryption |
| Typical Payload | "execute math.add" with args | "plan step 3", "critique score=7" |
| Latency Pattern | Synchronous per tool invocation | Potential network hops; can parallelize widely |
| Statefulness | Protocol stateless, tool may hold state | Agents hold memory; buses retain messages |
| Failure Modes | Invalid params, tool missing | Lost/dup messages, ordering issues, congestion |
| Complement Use | A2A messages request TOOL(...) | MCP tool responses fed back as new agent messages |
| Best Separation | Use MCP for *actionable capabilities* | Use A2A for *dialogue and coordination logic* |

### 6.10.1 Integration Flow

```mermaid
sequenceDiagram
    participant A2A as Agent Message Bus<br/>(A2A)
    participant Agent as Agent<br/>(Planner/Researcher)
    participant MCPC as MCP Client
    participant MCPS as MCP Server
    
    Agent->>A2A: envelope(content: "TOOL(add:{a:3,b:5})")
    A2A->>MCPC: detect TOOL(...) directive
    MCPC->>MCPS: tools/call(name: "add", arguments: {a:3,b:5})
    MCPS-->>MCPC: result({ok: true, result: 8})
    MCPC-->>A2A: wrap result in new envelope
    A2A->>Agent: envelope(content: "Tool result: 8")
```

### 6.10.2 Summary

- **MCP** = "How do I find and call capabilities safely?"
- **A2A** = "How do independent reasoning units collaborate over time?"
- **Together** = Tool-augmented multi-agent systems where agents coordinate (A2A) and invoke capabilities (MCP) seamlessly

### 6.10.3 Exercise Add-on

1. In the broker demo, allow messages starting with `TOOL(` to dispatch to the MCP-like Flask server and append the result as a follow-up envelope.
2. Add JSON Schema validation to envelope `content` when `type='tool_result'`.
3. Track how often tool invocations appear per round; compute tool utilization ratio.
